---
Ejercicio Integrador
---

In [6]:
import gradio as gr
from collections import Counter
from pyngrok import ngrok

In [7]:
class ProcesadorTexto:
    """Encapsula un texto y expone operaciones de análisis lingüístico básico."""

    def __init__(self, texto):
        # Nota sobre limpieza: strip() solo elimina caracteres en los extremos del string.
        # Para limpiar puntuación en todo el texto usamos replace(), igual que en limpiar_texto().
        print(f"Creando procesador para: '{texto[:40]}...'")
        self.texto_original = texto
        caracteres_a_eliminar = ['.', ',', '!', '?', ';', ':', '¿', '¡', '"', "'", '(', ')']
        texto_limpio = texto.lower()
        for char in caracteres_a_eliminar:
            texto_limpio = texto_limpio.replace(char, '')
        self.texto_limpio = texto_limpio
        self.palabras = self.texto_limpio.split()

    def contar_palabras(self):
        """Devuelve el total de palabras (con repeticiones)."""
        return len(self.palabras)

    def contar_palabras_unicas(self):
        """Devuelve la cantidad de palabras distintas. `set` elimina duplicados."""
        return len(set(self.palabras))

    def calcular_frecuencia(self):
        """Devuelve un diccionario con la frecuencia de cada palabra."""
        frecuencia = {}
        for palabra in self.palabras:
            frecuencia[palabra] = frecuencia.get(palabra, 0) + 1
        return frecuencia

    # Agregá el método calcular_longitud_media(self)
    def calcular_longitud_media(self):
        """Devuelve la longitud promedio de las palabras"""
        if not self.palabras:
            return 0.0
        return sum(len(palabra) for palabra in self.palabras) / len(self.palabras)
    

    def top_palabras(self, n=5):
        """Devuelve las n palabras más frecuentes como lista de tuplas (palabra, frecuencia)."""
        return Counter(self.palabras).most_common(n)
    
    def calcular_diversidad_lexica(self):
        """Devuelve el porcentaje de palabras únicas sobre el total (rango: 0 a 100)."""
        total = self.contar_palabras()
        if total == 0:
            return 0
        return (self.contar_palabras_unicas() / total) * 100

In [8]:
def analizar_texto(texto):
    """
    Núcleo de la aplicación.
    Recibe un string de texto, lo procesa con ProcesadorTexto y devuelve dos valores:
    - Un string con el resumen del análisis.
    - Un diccionario con la frecuencia de cada palabra.
    """

    if not texto.strip():
        return "Por favor, ingresá texto para analizar.", {}

    procesador = ProcesadorTexto(texto)

    total_palabras = procesador.contar_palabras()
    palabras_unicas = procesador.contar_palabras_unicas()
    frecuencias = procesador.calcular_frecuencia()

    if frecuencias:
        palabra_mas_frecuente = max(frecuencias, key=frecuencias.get)
        repeticiones = frecuencias[palabra_mas_frecuente]
    else:
        palabra_mas_frecuente = "N/A"
        repeticiones = 0

    resumen = (
        f"--- Resumen del Análisis ---\n"
        f"Total de palabras:     {total_palabras}\n"
        f"Palabras únicas:       {palabras_unicas}\n"
        f"Palabra más frecuente: '{palabra_mas_frecuente}' ({repeticiones} veces)"

        )
    
    return resumen, frecuencias

In [9]:
texto_de_prueba = """
Python es un lenguaje de programación versátil y poderoso.
Python se utiliza en muchas áreas, desde desarrollo web hasta ciencia de datos.
La simplicidad de Python lo hace ideal para principiantes,
pero su potencia satisface incluso a programadores expertos.
"""

In [10]:
resumen, frecuencias = analizar_texto(texto_de_prueba)
print(resumen)

Creando procesador para: '
Python es un lenguaje de programación v...'
--- Resumen del Análisis ---
Total de palabras:     39
Palabras únicas:       35
Palabra más frecuente: 'python' (3 veces)


In [14]:
# --- PASO 2: Configurá la Interfaz de Gradio ---
demo = gr.Interface(
    fn=analizar_texto,

    inputs=gr.Textbox(
        lines=10,
        placeholder="Escribí o pegá un texto acá para analizarlo...",
        label="Texto a analizar"
    ),

    outputs=[
        gr.Textbox(label="Resumen del Análisis"),
        gr.JSON(label="Frecuencia de Palabras")
    ],

    title="Analizador de Texto",
    description="Introducí un texto y obtené un análisis básico de PLN.",
    # allow_flagging="never"
)

# Descomentá la línea que corresponda a tu entorno:
# demo.launch()                # En Google Colab (crea una URL pública temporal con share=True)
# demo.launch(inbrowser=True)  # En entorno local: abre el navegador automáticamente
demo.launch(share=True, show_error=True)    # Crea un enlace público temporal (requiere internet)
#demo.launch(server_port=7862, share=False)
#public_url = ngrok.connect(7862)
#print(f"URL pública: {public_url}")


* Running on local URL:  http://127.0.0.1:7863

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


In [11]:
# Probá la función directamente, sin Gradio
texto_prueba = "Python es un lenguaje poderoso. Python se usa en PLN."
resumen_prueba, frecuencias_prueba = analizar_texto(texto_prueba)
print(resumen_prueba)
print(frecuencias_prueba)

Creando procesador para: 'Python es un lenguaje poderoso. Python s...'
--- Resumen del Análisis ---
Total de palabras:     10
Palabras únicas:       9
Palabra más frecuente: 'python' (2 veces)
{'python': 2, 'es': 1, 'un': 1, 'lenguaje': 1, 'poderoso': 1, 'se': 1, 'usa': 1, 'en': 1, 'pln': 1}
